# TraceRAG walkthrough: from raw text to a ranked list, step by step

This notebook opens up the sparse retriever for one SciFact test claim and recomputes every number by hand:
tokens, dictionary entries, postings, the **ltc** query vector, the **lnc** document vector, the cosine score,
the zone combination, and finally BM25, dense and hybrid rankings. Each hand calculation is checked against
the library code with an `assert`.

Run `python scripts/build_index.py --dense` first (from the project root).

In [1]:
import math, os, sys, warnings
from collections import Counter
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import config
from src.index.inverted_index import get_index
from src.index.tokenizer import Tokenizer
from src.ingest.load_scifact import load_queries, load_qrels
from src.retrieve.tfidf import TfidfRetriever
from src.retrieve.bm25 import BM25Retriever

pd.set_option("display.max_colwidth", 80)
queries, qrels = load_queries(), load_qrels("test")
QID = "53"
claim = queries[QID]["text"]
print("claim", QID, ":", claim)
print("SciFact label:", queries[QID]["label"], "| relevant papers:", sorted(qrels[QID]))

claim 53 : ALDH1 expression is associated with poorer prognosis in breast cancer.
SciFact label: SUPPORT | relevant papers: ['45638119']


## 1. Tokenizer

Case folding, punctuation stripping, stopword removal (NLTK list) and Porter stemming. Each step can be
switched off, which is how the tokenizer ablation in `results/retrieval_ablations.csv` was produced.

In [2]:
rows = []
for stem in (False, True):
    for stop in (False, True):
        rows.append({"stemming": stem, "stopwords removed": stop,
                     "tokens": " ".join(Tokenizer(stem=stem, stopwords=stop).tokenize(claim))})
pd.DataFrame(rows)

,stemming,stopwords removed,tokens
0,False,False,aldh1 expression is associated with poorer prognosis in breast cancer
1,False,True,aldh1 expression associated poorer prognosis breast cancer
2,True,False,aldh1 express i associ with poorer prognosi in breast cancer
3,True,True,aldh1 express associ poorer prognosi breast cancer


## 2. Dictionary and postings

The index stores, per zone, a sorted dictionary (term -> df and an offset into the postings arrays) and the
postings themselves (chunk, tf). Below: index size, then the dictionary entry and postings for one word.

In [3]:
index = get_index("whole")
N = index.n_chunks
print(index.stats())
index.print_postings("tumor", zone="body", limit=5);

{'title': {'terms': 8092, 'postings': 50806, 'avg_len': 10.05}, 'body': {'terms': 26041, 'postings': 461284, 'avg_len': 141.34}, 'all': {'terms': 26321, 'postings': 469814, 'avg_len': 151.39}, 'chunks': 5183, 'papers': 5183, 'tokenizer': {'stem': True, 'stopwords': True}}
word 'tumor' -> index term 'tumor'   zone=body   tokenizer=Tokenizer(stem=True, stopwords=True)
  df = 535 of N = 5183 chunks   idf = log10(N/df) = 0.9862
  postings (top 5 by tf; stored sorted by chunk):
          25726838  tf=19  len=169  IL-17 promotes tumor development through the induction of tu
          30919024  tf=16  len=198  Molecular mechanisms underlying tumor dormancy.
           4489217  tf=13  len=198  Intratumor heterogeneity and branched evolution revealed by 
          23420807  tf=13  len=231  Anti-Angiogenic Activity of Selected Receptor Tyrosine Kinas
           7986878  tf=12  len=169  Anti-alphav integrin monoclonal antibody intetumumab enhance


In [4]:
q_terms = Tokenizer().tokenize(claim)
body = index.zones["body"]
pd.DataFrame([{"term": t, "df (body)": body.doc_freq(t), "idf = log10(N/df)": round(body.idf(t), 4),
               "postings (first 3)": index.get_postings(t, "body")[:3]} for t in dict.fromkeys(q_terms)])

,term,df (body),idf = log10(N/df),postings (first 3)
0,aldh1,1,3.7146,"[(45638119, 1)]"
1,express,1516,0.5339,"[(18670, 1), (19238, 4), (33370, 2)]"
2,associ,1502,0.5379,"[(33370, 1), (92308, 1), (97884, 3)]"
3,poorer,18,2.4593,"[(750781, 1), (5372432, 1), (6421734, 3)]"
4,prognosi,82,1.8008,"[(427865, 1), (515489, 1), (601033, 1)]"
5,breast,177,1.4666,"[(313403, 8), (756887, 3), (878526, 1)]"
6,cancer,790,0.8170,"[(5836, 1), (33370, 1), (213017, 1)]"


## 3. Query vector: ltc

`w(t,q) = (1 + log10 tf) * log10(N/df)`, then divide by the vector's length. Shown for the body zone.

In [5]:
tf_q = Counter(q_terms)
qrows = []
for t, tf in tf_q.items():
    df = body.doc_freq(t)
    if df == 0:
        continue
    qrows.append({"term": t, "tf": tf, "1+log10 tf": 1 + math.log10(tf), "df": df,
                  "idf": math.log10(N / df), "raw w": (1 + math.log10(tf)) * math.log10(N / df)})
q_norm = math.sqrt(sum(r["raw w"] ** 2 for r in qrows))
for r in qrows:
    r["w (normalised)"] = r["raw w"] / q_norm
qdf = pd.DataFrame(qrows).round(4)
retriever = TfidfRetriever(index, (0.0, 1.0))      # body zone only, to match this hand calculation
lib = retriever.query_weights(claim, "body")
assert all(abs(lib[r["term"]] - r["w (normalised)"]) < 1e-12 for r in qrows)
print("query vector length before normalising:", round(q_norm, 4))
qdf

query vector length before normalising: 5.146


,term,tf,1+log10 tf,df,idf,raw w,w (normalised)
0,aldh1,1,1.0,1,3.7146,3.7146,0.7218
1,express,1,1.0,1516,0.5339,0.5339,0.1037
2,associ,1,1.0,1502,0.5379,0.5379,0.1045
3,poorer,1,1.0,18,2.4593,2.4593,0.4779
4,prognosi,1,1.0,82,1.8008,1.8008,0.3499
5,breast,1,1.0,177,1.4666,1.4666,0.2850
6,cancer,1,1.0,790,0.8170,0.8170,0.1588


## 4. Document vector: lnc, and the cosine score

`w(t,d) = (1 + log10 tf) / ||d||` with no idf; `||d||` runs over *all* terms of the abstract, not only the query
terms. The score is the dot product over the shared terms.

In [6]:
top_id, top_score = retriever.search(claim, 1)[0]
c = index.chunk_pos[top_id]
doc_tf = Counter(index.tokenizer.tokenize(index.chunks[c].text))
d_norm = math.sqrt(sum((1 + math.log10(v)) ** 2 for v in doc_tf.values()))
assert abs(d_norm - body.lnc_norm[c]) < 1e-9
drows, score = [], 0.0
for r in qrows:
    tf = doc_tf.get(r["term"], 0)
    wd = (1 + math.log10(tf)) / d_norm if tf else 0.0
    score += r["w (normalised)"] * wd
    drows.append({"term": r["term"], "tf in doc": tf, "w(t,d)": wd, "w(t,q)": r["w (normalised)"],
                  "product": r["w (normalised)"] * wd})
print(f"top paper {top_id}: {index.chunks[c].title}")
print(f"document length ||d|| = {d_norm:.4f} over {len(doc_tf)} distinct terms")
print(f"hand-computed cosine = {score:.6f}   library score = {top_score:.6f}")
assert abs(score - top_score) < 1e-12
pd.DataFrame(drows).round(4)

top paper 45638119: ALDH1 is a marker of normal and malignant human mammary stem cells and a predictor of poor clinical outcome.
document length ||d|| = 9.3055 over 74 distinct terms
hand-computed cosine = 0.200553   library score = 0.200553


,term,tf in doc,"w(t,d)","w(t,q)",product
0,aldh1,1,0.1075,0.7218,0.0776
1,express,1,0.1075,0.1037,0.0111
2,associ,0,0.0000,0.1045,0.0000
3,poorer,0,0.0000,0.4779,0.0000
4,prognosi,1,0.1075,0.3499,0.0376
5,breast,5,0.1826,0.2850,0.0520
6,cancer,2,0.1398,0.1588,0.0222


## 5. Zones: title vs body

The final tf-idf score is `(w_title * cos_title + w_body * cos_body) / (w_title + w_body)`. The title weight was
chosen on the train claims (see `results/tuned_params.json`); high title weights hurt because a short title
vector gives large cosines from a single matching word.

In [7]:
from src.retrieve.factory import zone_weights_for
w = zone_weights_for("whole")
zoned = TfidfRetriever(index, w)
cos_t, cos_b = zoned.zone_scores(claim, "title"), zoned.zone_scores(claim, "body")
final = zoned.score_all(claim)
rows = []
for cid, s in zoned.search(claim, 5):
    i = index.chunk_pos[cid]
    rows.append({"paper": cid, "relevant": cid in qrels[QID], "cos_title": cos_t[i], "cos_body": cos_b[i],
                 "final": s, "title": index.chunks[i].title[:70]})
    assert abs((w[0] * cos_t[i] + w[1] * cos_b[i]) / (w[0] + w[1]) - s) < 1e-12
print("zone weights (title, body):", w)
pd.DataFrame(rows).round(4)

zone weights (title, body): (0.1, 1.0)


,paper,relevant,cos_title,cos_body,final,title
0,45638119,True,0.1707,0.2006,0.1978,ALDH1 is a marker of normal and malignant human mammary stem cells and
1,25973484,False,0.2287,0.1479,0.1552,Breast cancer and obesity: an update.
2,7821634,False,0.1321,0.1348,0.1346,Profiling of residual breast cancers after neoadjuvant chemotherapy id
3,21439640,False,0.1942,0.1208,0.1275,Macrophages induce COX-2 expression in breast cancer cells: role of IL
4,5372432,False,0.0588,0.1286,0.1222,Geographical access to healthcare in Northern England and post-mortem


## 6. Why did the top paper win? Term contributions

In [8]:
from src.explain.contrib import top_terms
pd.DataFrame(top_terms(claim, rows[0]["paper"], zoned, n=8)).round(4)

,term,zone,tf,df,w_query,w_doc,contribution,share
0,aldh1,body,1,1,0.7218,0.1075,0.0705,0.3564
1,breast,body,5,177,0.2850,0.1826,0.0473,0.2391
2,prognosi,body,1,82,0.3499,0.1075,0.0342,0.1728
3,cancer,body,2,790,0.1588,0.1398,0.0202,0.1020
4,aldh1,title,1,1,0.5912,0.2887,0.0155,0.0784
5,express,body,1,1516,0.1037,0.1075,0.0101,0.0512


## 7. BM25 on the same claim

In [9]:
bm25 = BM25Retriever(index)
pd.DataFrame([{"rank": r + 1, "paper": d, "relevant": d in qrels[QID], "BM25": round(s, 3),
               "title": index.chunks[index.chunk_pos[d]].title[:70]}
              for r, (d, s) in enumerate(bm25.search(claim, 5))])

,rank,paper,relevant,BM25,title
0,1,45638119,True,27.048,ALDH1 is a marker of normal and malignant human mammary stem cells and
1,2,25973484,False,18.547,Breast cancer and obesity: an update.
2,3,7821634,False,18.277,Profiling of residual breast cancers after neoadjuvant chemotherapy id
3,4,21439640,False,16.566,Macrophages induce COX-2 expression in breast cancer cells: role of IL
4,5,5372432,False,16.331,Geographical access to healthcare in Northern England and post-mortem


## 8. Dense and hybrid

Needs the MiniLM model (downloaded automatically on first use).

In [10]:
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()
from src.retrieve.factory import RetrieverFactory
f = RetrieverFactory()
out = []
for name in ("tfidf", "bm25", "dense", "hybrid_rrf"):
    ranked = [d for d, _ in f.get(name, "whole").rank_docs(claim, 100)]
    first = next((i + 1 for i, d in enumerate(ranked) if d in qrels[QID]), None)
    out.append({"retriever": name, "top 3": ranked[:3], "rank of first relevant paper": first})
pd.DataFrame(out)

,retriever,top 3,rank of first relevant paper
0,tfidf,"[45638119, 25973484, 7821634]",1
1,bm25,"[45638119, 25973484, 7821634]",1
2,dense,"[45638119, 32001951, 17648235]",1
3,hybrid_rrf,"[45638119, 7821634, 29366489]",1


## 9. The evaluation tables

Produced by `python -m src.eval.run_retrieval_eval` on the 300 test claims.

In [11]:
main = Path("results/retrieval_main.csv")
pd.read_csv(main).round(3) if main.exists() else print("run python -m src.eval.run_retrieval_eval first")

,Retriever,P@5,P@10,R@10,MRR@10,nDCG@10,nDCG@10 95% CI
0,tf-idf lnc.ltc (zones),0.163,0.093,0.846,0.644,0.687,0.646–0.728
1,BM25,0.162,0.091,0.820,0.644,0.683,0.639–0.725
2,Dense (MiniLM),0.164,0.088,0.783,0.605,0.645,0.599–0.691
3,Hybrid RRF (BM25 + dense),0.175,0.096,0.858,0.683,0.720,0.677–0.761
4,Hybrid weighted (BM25 + dense),0.174,0.096,0.857,0.700,0.733,0.691–0.774
